# Prototype V1: Semantic Segmentation model
Modelling for green surface detection via NDVI mask and building mask raster calculations. The actual raster calculations can be run in an additional Jupyter notebook (raster_calculations.ipynb):  
- Building mask: Rasterizing the vector layer of building footprints
- Vegetation mask: Creating a mask based on a NDVI threshold (still need to research on the optimal threshold for this !!!!)  
- Multiplication Building mask * vegetation mask = only vegetation on buildings  
- Addition only vegetation on buildings mask + building mask  

This notebook is the prototype running only on a subset from the originally chosen 1km x 1km tile.  

----
#### Overall Idea/Concept
Input: Aerial vexcel imagery  
Labels: Combined raster of building and vegetation mask  
Output: a pixel-wise map of green roofs/vegetated roof areas  

---
#### Current possible limitations *(still need to be checked !!!)*
- overhaning trees  
- slight raster alignment errors  
=> how will i deal with this?

In [2]:
# loading all required libraries
from pathlib import Path
import geoai
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import numpy as np

In [ ]:
# define path !!!!!!!
# need to run the generate training data notebook first to generate the training data


# Verify masks
with rasterio.open("mask_subset.tif") as src:
    mask = src.read(1)

print(np.unique(mask))

In [ ]:
# Check Class balance
unique, counts = np.unique(mask, return_counts=True)

for u, c in zip(unique, counts):
    print(u, c)

In [ ]:
# inspect raster and reference polygons
geoai.print_raster_info(
    train_raster_path,
    show_preview=False,
)

In [ ]:
train_vector_gdf = gpd.read_file(train_vector_path)
for col in train_vector_gdf.columns:
    if train_vector_gdf[col].dtype.kind == "M":  # datetime64, incl. timezone-aware
        train_vector_gdf[col] = train_vector_gdf[col].astype(str)

geoai.view_vector_interactive(
    train_vector_gdf,
    tiles=train_raster_path,
)

In [ ]:
geoai.view_raster(test_raster_path)

In [ ]:
# Create training tiles
out_folder = "buildings"

tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)

In [ ]:
# Train U-Net model
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

In [ ]:
# Evaluate Model
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
)

In [ ]:
# Run inference on test raster
masks_path = "marly_buildings_prediction.tif"
probability_path = "marly_test_probability_map.tif"
model_path = f"{out_folder}/unet_models/best_model.pth"

geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    probability_path=probability_path,
)

In [ ]:
# Inspect masks and ucertainty
geoai.view_raster(
    masks_path,
    nodata=0,
    colormap="binary",
    basemap=test_raster_path,
)

In [ ]:
geoai.view_raster(
    probability_path,
    indexes=[2],
    basemap=test_raster_path,
)

In [ ]:
# Vectorize and filter predictions
output_vector_path = "marly_buildings_prediction.geojson"

gdf = geoai.orthogonalize(
    masks_path,
    output_vector_path,
    epsilon=2,
)

gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")
gdf_filtered = gdf_props[(gdf_props["area_m2"] > 10)]

geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_path,
)